# Clean random rewire — removing a 4.3% contamination

`build_directionality_variants.py` draws random parents from `basin_arr[basin_arr != c]`: it excludes
the basin itself but **not its true parents**, so 27 of 624 random edges (4.3%, affecting 23 of 150
connected basins) are real edges recurring by chance.

Under the paper's current framing this matters more than it used to. The random rewire is no longer
a topology control, it is the **far end of the distance axis** (~511 km). Recurring true parents are
*nearby* (~92 km), so the contamination injects short-distance edges into the long-distance arm and
biases it upward: the true distance decay is steeper than reported.

This notebook rebuilds the random control with true parents excluded and retrains it at three seeds.

Pre-registration: `experiments/topology_ablation/preregistration_clean_random.md`.

**Runtime → Change runtime type → T4 GPU → Run all.** ~2 h.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config (SEEDS = [11, 13, 17])

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEEDS=[11,13,17]
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydro_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('CAMELS:', DRIVE_CAMELS_PATH); print('RUNS  :', DRIVE_RUNS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(f'{DRIVE_RUNS}/topology_ablation/component0', exist_ok=True)
print('datasets ->', os.path.realpath(RD)); print('runs     ->', os.path.realpath(RR))

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Rebuild the random control, excluding true parents

Same in-degree-preserving draw, same RNG seed, but the forbidden set now includes each basin's true
parents (mirroring `build_distance_control.py`). We print the overlap of both the old and new graphs
so the correction is auditable.

In [ ]:
%cd {REPO_DIR}
import pickle, numpy as np, pandas as pd, networkx as nx
from pathlib import Path
FEAT='experiments/topology_ablation/features'
P1=Path('topology_analysis/phase1_network_discovery/outputs')
TOPO_TXT='datasets/camels_us/camels_attributes_v2.0/camels_topo.txt'
RNG_SEED=42
def named_ok(p, n_expected=183):
    # must exist, have a 'date'-named index, AND cover every basin (NH KeyErrors on a missing one)
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True

basins=[l.strip() for l in open(P1/'component0_basins.txt') if l.strip()]
E=pd.read_csv(P1/'component0_edges.csv',dtype={'parent_id':str,'child_id':str})
fwd=list(zip(E.parent_id,E.child_id))
true_parents={b:[] for b in basins}
indeg={b:0 for b in basins}
for p,c_ in fwd:
    true_parents[c_].append(p); indeg[c_]+=1
true_set=set(fwd)

def draw(exclude_true):
    rng=np.random.default_rng(RNG_SEED); arr=np.array(basins); out=[]
    for c_ in basins:
        k=indeg[c_]
        if k==0: continue
        forb={c_}|(set(true_parents[c_]) if exclude_true else set())
        avail=np.array([b for b in arr if b not in forb])
        pick=rng.choice(avail,size=min(k,len(avail)),replace=False)
        for p in pick: out.append((str(p),c_))
    return out
old=draw(False); new=draw(True)
print(f'OLD random: {len(old)} edges | overlap with true = {sum(1 for e in old if e in true_set)} '
      f'({100*sum(1 for e in old if e in true_set)/len(old):.1f}%)')
print(f'NEW random: {len(new)} edges | overlap with true = {sum(1 for e in new if e in true_set)} '
      f'({100*sum(1 for e in new if e in true_set)/len(new):.1f}%)')
pd.DataFrame(new,columns=['parent_id','child_id']).to_csv(P1/'component0_edges_randomclean.csv',index=False)
print('wrote', P1/'component0_edges_randomclean.csv')

## Cell 8 — Build the clean-random feature

In [ ]:
%cd {REPO_DIR}
out=f'{FEAT}/upstream_q_randomclean_component0_lag1.p'
if named_ok(out):
    print('feature present, skipping')
else:
    from neuralhydrology.datasetzoo.camelsus import load_camels_us_discharge
    topo=pd.read_csv(TOPO_TXT,sep=';',dtype={'gauge_id':str}).set_index('gauge_id')
    area={b:float(topo.loc[b,'area_gages2']) for b in basins}
    q={}
    for b in basins:
        try: q[b]=load_camels_us_discharge(Path('datasets/camels_us'),b,area[b])
        except Exception: q[b]=None
    par={b:[] for b in basins}
    for p,c_ in new: par[c_].append(p)
    feats={}
    for c_ in basins:
        ps=par[c_]
        if not ps: continue
        idx=None
        for p in ps:
            if q.get(p) is not None: idx=q[p].index; break
        if idx is None: continue
        agg=pd.Series(0.0,index=idx); wsum=0.0
        for p in ps:
            if q.get(p) is None: continue
            agg=agg.add((q[p].reindex(idx)*area[p]).fillna(0.0),fill_value=0.0); wsum+=area[p]
        if wsum<=0: continue
        s=(agg/wsum).shift(1).fillna(0.0)
        feats[c_]=pd.DataFrame({'upstream_q':s.values},index=pd.DatetimeIndex(idx,name='date'))
    pickle.dump(feats,open(out,'wb'))
    print(f'built {len(feats)} basins')
print('date-named index:', named_ok(out))

## Cell 9 — Train the clean random control at three seeds

In [ ]:
%cd {REPO_DIR}
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
for s in SEEDS:
    if done('L_upQrandclean',s): print(f'seed {s}: done'); continue
    print(f'=== training L_upQrandclean seed {s} ===')
    !python experiments/topology_ablation/run_upstream_feature.py \
        --network component0 --seed {s} --device cuda:0 --epochs 30 \
        --feature-file experiments/topology_ablation/features/upstream_q_randomclean_component0_lag1.p \
        --cond-name L_upQrandclean

## Cell 10 — Verdict: does the far end of the distance axis move?

In [ ]:
%cd {REPO_DIR}
import pandas as pd, numpy as np, pickle, glob
from scipy.stats import wilcoxon
FEAT='experiments/topology_ablation/features'
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def named_ok(p, n_expected=183):
    # must exist, have a 'date'-named index, AND cover every basin (NH KeyErrors on a missing one)
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')
def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    return pd.read_csv(p,dtype={'basin':str}).set_index('basin')['NSE'] if os.path.isfile(p) else None
_f=pickle.load(open(f'{FEAT}/upstream_q_component0_lag1.p','rb'))
CONN=sorted([b for b,v in _f.items() if float(np.nanmax(np.abs(v.values)))>0])
def paired(cond,s,ref='L',basins=None):
    A=nse(cond,s); L=nse(ref,s)
    if A is None or L is None: return None
    bs=[b for b in (basins or CONN) if b in A.index and b in L.index]
    return (A[bs]-L[bs]).values
print('helpers ready | connected basins:', len(CONN))

In [ ]:
old_per=[]; new_per=[]
for s in SEEDS:
    d=paired('L_upQrand',s)
    if d is not None: old_per.append(np.median(d))
    d=paired('L_upQrandclean',s)
    if d is not None: new_per.append(np.median(d))
print('| control | per-seed Δ | cross-seed mean |')
print('|---|---|---|')
if old_per: print(f'| random (contaminated) | {[f"{x:+.3f}" for x in old_per]} | {np.mean(old_per):+.4f} |')
if new_per: print(f'| random (clean)        | {[f"{x:+.3f}" for x in new_per]} | {np.mean(new_per):+.4f} |')
if old_per and new_per:
    sh=np.mean(new_per)-np.mean(old_per)
    print(f'\nshift from removing 4.3% true-edge contamination: {sh:+.4f}')
    print('\n=== VERDICT ===')
    if sh < -0.002:
        print('  As predicted: the clean control is LOWER. The contamination was inflating the far end,')
        print('  so the true distance decay is steeper than the paper reports. Update the random row.')
    elif abs(sh) <= 0.002:
        print('  No material change. The contamination was immaterial; report the clean number and')
        print('  note the old one was within noise of it.')
    else:
        print('  Clean control is HIGHER, which is unexpected. Inspect before writing.')

## Cell 11 — Persistence check

In [ ]:
print('=== persistence (in Drive?) ===')
for s in SEEDS:
    dp=f'{DRIVE_RUNS}/topology_ablation/component0/L_upQrandclean_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    print(f'  L_upQrandclean seed {s}: {os.path.isfile(dp)}')

## Done

Report **Cell 7 overlap counts** and the **Cell 10 verdict**. This removes a known defect from a
control the paper relies on, and it sharpens the distance story either way.